# Imports

In [35]:
import os
from scapy.all import rdpcap, wrpcap
from collections import Counter
from datetime import datetime

# Récupération des données

In [36]:
attack_dir = 'raw/attack_pcap'
benign_dir = 'raw/benign_pcap'

processed_dir = 'raw/processed'


In [39]:
def merge_pcaps(directory, output):
        '''
        Merge all pcap files in a directory into a single pcap file.
        Args:
                directory (str): Path to the directory containing pcap files.
                output (str): Path to the output pcap file.
        Returns:
                list: List of packets in the merged pcap file.
        '''
        packets = []
        files = sorted([f for f in os.listdir(directory) if f.endswith(".pcap")])

        for f in files:
                packets.extend(rdpcap(os.path.join(directory, f)))

        packets.sort(key=lambda p: p.time)

        output = output.replace(".pcap", f'_{datetime.now().strftime("%Y-%m-%d_%H-%M-%S")}.pcap')
        os.makedirs(processed_dir, exist_ok=True)
        wrpcap(output, packets)        
        return packets

In [42]:
attack_packets = merge_pcaps(attack_dir, processed_dir + '/merged_attack.pcap')
benign_packets = merge_pcaps(benign_dir, processed_dir + '/merged_benign.pcap')

# Statistiques sur les fichiers pcap assemblés

In [ ]:

from scapy.layers.inet import IP

# Mapping IANA des numéros de protocoles vers leurs noms les plus courants
PROTO_NAMES = {
    # --- Common / Core ---
    0:  "HOPOPT",                 # IPv6 Hop-by-Hop Options
    1:  "ICMP",
    2:  "IGMP",
    4:  "IPv4",                   # IP-in-IP (encapsulation of IPv4)
    6:  "TCP",
    8:  "EGP",
    9:  "IGP",                    # Interior Gateway Protocol (old)
    17: "UDP",
    41: "IPv6",                   # IPv6 encapsulation
    43: "IPv6-Route",             # Routing Header for IPv6
    44: "IPv6-Frag",              # Fragment Header for IPv6
    47: "GRE",
    50: "ESP",                    # IPsec Encapsulating Security Payload
    51: "AH",                     # IPsec Authentication Header
    58: "ICMPv6",
    59: "IPv6-NoNxt",             # No Next Header
    60: "IPv6-Opts",              # Destination Options for IPv6
    88: "EIGRP",
    89: "OSPF",
    94: "IPIP-Encap",             # (used by some stacks for IP-within-IP variants)
    97: "ETHERIP",
    98: "ENCAP",                  # Yet Another IP encapsulation
    103:"PIM",                    # Protocol Independent Multicast
    108:"IPComp",                 # IP Payload Compression
    112:"VRRP",
    115:"L2TP",
    132:"SCTP",
    136:"UDPLite",

    # --- Additional transports / less common but useful ---
    27: "RDP",                    # Reliable Datagram Protocol (historic)
    33: "DCCP",
    46: "RSVP",
    66: "RVD",                    # MIT Remote Virtual Disk (historic)
    73: "RUDP",                   # Reliable UDP (rare/experimental in the wild)
    77: "SUN-ND",                 # Sun Network Disk (historic)
    80: "MTP",                    # Multicast Transport Protocol (historic)

    # --- Routing / control / discovery ---
    85: "MHRP",                   # Mobile Host Routing Protocol (historic)
    88: "EIGRP",
    89: "OSPF",
    90: "MPLS-in-IP",             # (a.k.a. "sprite-rpc" historically; MPLS often seen as 137/ISO?) -> keep as label used in some tools
    94: "IPIP-Encap",
    96: "IP-MicroMobility",       # (several experimental mobility headers)
    98: "ENCAP",
    103:"PIM",
    112:"VRRP",
    124:"IS-IS over IPv4",        # IS-IS over IPv4 (not common)

    # --- Tunneling / encapsulations ---
    47: "GRE",
    50: "ESP",
    51: "AH",
    97: "ETHERIP",
    98: "ENCAP",
    115:"L2TP",
    137:"MPLS-in-UDP",            # MPLS over UDP (used in some deployments)

    # --- IPv6 related extras ---
    0:  "HOPOPT",
    43: "IPv6-Route",
    44: "IPv6-Frag",
    58: "ICMPv6",
    59: "IPv6-NoNxt",
    60: "IPv6-Opts",

    # --- Multicast / streaming ---
    33: "DCCP",
    103:"PIM",
    113:"PGM",                    # Pragmatic General Multicast

    # --- Security / auth / compression ---
    50: "ESP",
    51: "AH",
    108:"IPComp",

    # --- Vendor/legacy/rare (kept for completeness if you parse old pcaps) ---
    3:  "GGP",
    5:  "ST",
    7:  "CBT",
    10: "BBN-RCC-MON",
    11: "NVP-II",
    12: "PUP",
    13: "ARGUS",
    14: "EMCON",
    15: "XNET",
    16: "CHAOS",
    18: "MUX",
    19: "DCN-MEAS",
    20: "HMP",
    21: "PRM",
    22: "XNS-IDP",
    23: "TRUNK-1",
    24: "TRUNK-2",
    25: "LEAF-1",
    26: "LEAF-2",
    28: "IRTP",
    29: "ISO-TP4",
    30: "NETBLT",
    31: "MFE-NSP",
    32: "MERIT-INP",
    34: "TPC",        # Third Party Connect (historic)
    35: "IDPR",
    36: "XTP",
    37: "DDP",        # Datagram Delivery Protocol (AppleTalk)
    38: "IDPR-CMTP",
    39: "TP++",
    40: "IL",
    42: "SDRP",
    45: "IDRP",
    48: "IGP-PVT",    # Private Interior Gateway (historic)
    49: "SPI-NET",
    52: "NARP",
    54: "SWIPE",      # IP with Encryption (old)
    55: "NARP-Alt",   # (variant labels in some stacks)
    57: "SKIP",
    61: "Any-Host-Internal",
    62: "CFTP",
    63: "Any-Local-Network",
    64: "SAT-EXPAK",
    65: "KRYPTOLAN",
    67: "IPPC",
    69: "SAT-MON",
    70: "VISA",
    71: "IPCV",
    72: "CPNX",
    74: "IRTP-Alt",
    75: "ISO-IP",
    76: "VMTP",
    78: "TP+",
    79: "HMP-Alt",
    81: "SECURE-VMTP",
    82: "VINES",
    83: "TTP/IPTM",
    84: "NSFNET-IGP",
    86: "TLSP",
    87: "EIGRP-Old",
    91: "OSPF-Any",
    92: "MTP-Alt",
    93: "AX.25",
    95: "SATNET-MON",
    99: "GMTP",
    100:"IFMP",
    101:"PNNI",
    102:"PIM-Alt",
    104:"ARIS",
    105:"SCPS",
    106:"QNX",
    107:"A/N",
    109:"SNP",
    110:"Compaq-Peer",
    111:"IPX-in-IP",
    113:"PGM",
    114:"0-HOP",
    116:"DDX",
    117:"IATP",
    118:"STP",
    119:"SRP",
    120:"UTI",
    121:"SMP",
    122:"SM",
    123:"PTP",
    125:"FIRE",
    126:"CRTP",
    127:"CRUDP",
    128:"SSCOPMCE",
    129:"IPLT",
    130:"SPS",
    131:"PIPE",
    133:"FC",         # Fibre Channel
    134:"RSVP-E2E-IGNORE",
    135:"Mobility Header",
    137:"MPLS-in-UDP",
    138:"MANET",
    139:"HIP",
    140:"Shim6",
    141:"WESP",
    142:"ROHC",
    # 143-252: divers/expérimentaux/réservés – laissés à fallback
    255:"Reserved"    # (souvent utilisé comme "Reserved"/"RAW")
}

def get_proto_name(pkt):
    """
    Retourne le nom du protocole (ex: 'TCP') si possible.
    Fallback: 'proto:<num>' si inconnu, 'unknown' si non applicable.
    """
    # Si c'est un paquet IP (v4)
    if pkt.haslayer(IP):
        num = pkt[IP].proto
        return PROTO_NAMES.get(num, f"proto:{num}")
    # Certains paquets peuvent être IPv6 (si tu utilises scapy.layers.inet6)
    try:
        from scapy.layers.inet6 import IPv6
        if pkt.haslayer(IPv6):
            num = pkt[IPv6].nh  # champ Next Header
            return PROTO_NAMES.get(num, f"proto:{num}")
    except Exception:
        pass
    return "unknown"

# --- Impression pour les paquets d'attaque ---
if attack_packets:
    print(f"Nombre de paquets d'attaque : {len(attack_packets)}")
    proto_counts_attack = Counter(get_proto_name(pkt) for pkt in attack_packets)
    print(f"Types de protocoles d'attaque : {proto_counts_attack}")
else:
    print("Aucun paquet d'attaque trouvé.")

# --- Impression pour les paquets bénins ---
if benign_packets:
    print(f"Nombre de paquets bénins : {len(benign_packets)}")
    proto_counts_benign = Counter(get_proto_name(pkt) for pkt in benign_packets)
    print(f"Types de protocoles bénins : {proto_counts_benign}")
else:
    print("Aucun paquet bénin trouvé.")


Nombre de paquets d'attaque : 3804
Types de protocoles d'attaque : Counter({'UDP': 2864, 'unknown': 848, 'TCP': 92})
Nombre de paquets bénins : 16730
Types de protocoles bénins : Counter({'TCP': 11051, 'unknown': 4565, 'ICMP': 742, 'UDP': 372})


: 